# 04 Vault writer

**Graduates to:** `hub/src/hudhud_hub/vault.py`

Every book with highlights gets a note in your Obsidian vault,
`Vault/Hudhud/<Title>.md`. SQLite stays the source of truth; the note is a
generated view. The trick is sharing one file with a human: Hudhud owns only
the text between two HTML comments, and everything else in the file is yours.

```markdown
---
hudhud_book_id: 3f2a9c...
title: "Thinking, Fast and Slow"
...
---

# Thinking, Fast and Slow

Anything written outside the markers is yours. Hudhud never changes it.

<!-- hudhud:start (managed by Hudhud; edit highlights in the app) -->
## Highlights

> The confidence people have in their intuitions is not a reliable guide...

^h-01jb7x3k9q2m4n5p6r7s8t9v0w

*Comment:* Compare with Taleb on narrative fallacy.
*Location:* 42% · [Open in reader](https://hub/read/3f2a9c...?h=01JB7X3K9Q2M4N5P6R7S8T9V0W)
<!-- hudhud:end -->
```

## The rules, and why each exists

- **Block IDs.** `^h-<lowercase ULID>` lets you link a single highlight from
  any note: `[[Thinking, Fast and Slow#^h-01jb...]]`. For a quote, Obsidian
  only recognises the ID on **its own line, with a blank line before and
  after**. Inside the `>` it would not work.
- **Order by position**, not by creation time, so the note reads like the book.
- **Frontmatter values are JSON-quoted.** A title like `Re: Kahneman` would
  otherwise be invalid YAML. A JSON string is always a valid YAML string.
- **Escape the markers.** If a highlight contains `<!-- hudhud:end -->` (a
  book about Hudhud, say), it would end the region early. Replace `<!--` with
  `&lt;!--` in highlight text and comments.
- **Filenames.** Strip `\ / : * ? " < > |` (Windows), `# ^ [ ]` (Obsidian
  links) and control characters. Trim trailing dots and spaces (Windows drops
  them silently), and append `_` to Windows reserved names like `CON`. Farsi
  titles stay as they are. Cap the name at 150 **bytes** of UTF-8, not 150
  characters: Linux allows 255 bytes per filename, a Farsi letter takes 2
  bytes and an emoji 4, and the name still needs room for
  ` (3f2a9c00).conflict-20261008T123456Z.md`. Cut on a character boundary.
- **Same title, different book.** If `<Title>.md` already belongs to another
  book (its frontmatter `hudhud_book_id` differs), use
  `<Title> (<first 8 of id>).md`.
- **Atomic writes.** Write a temporary file in the same folder, then
  `os.replace` it over the target. Obsidian never sees a half-written note.
  On Windows, `os.replace` can fail briefly while another program holds the
  file, so retry a few times.
- **Edit protection.** The hub remembers a hash of the region it last wrote.
  If the region on disk has a different hash, you edited it in Obsidian, so
  copy the whole file to `<Title>.conflict-<UTC stamp>.md` before
  overwriting. Normalise `\r\n` to `\n` before hashing, or an editor that
  saves Windows line endings would trigger a conflict on every write.
  Strip a leading byte-order mark (`\ufeff`) for the same reason: some
  editors add one, and it would also hide the frontmatter's book id.
- **Stay inside the vault.** Resolve every path and check it is still under
  the vault root.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-or-later
import hashlib
import json
import os
import re
import shutil
import tempfile
import time
from dataclasses import dataclass
from pathlib import Path

import ipytest

ipytest.autoconfig(raise_on_error=os.environ.get("HUDHUD_NB_CHECK") == "1")

START = "<!-- hudhud:start (managed by Hudhud; edit highlights in the app) -->"
START_PREFIX = "<!-- hudhud:start"
END = "<!-- hudhud:end -->"
INTRO = "Anything written outside the markers is yours. Hudhud never changes it."
FORBIDDEN = re.compile(r'[\\/:*?"<>|#^\[\]\x00-\x1f]')
WINDOWS_RESERVED = {"CON", "PRN", "AUX", "NUL",
                    *(f"COM{i}" for i in range(1, 10)), *(f"LPT{i}" for i in range(1, 10))}
MAX_TITLE_BYTES = 150


@dataclass(frozen=True)
class BookMeta:
    id: str
    title: str
    author: str | None
    language: str | None


@dataclass(frozen=True)
class Highlight:
    id: str
    text: str
    comment: str
    fraction: float
    created_at: str


@dataclass(frozen=True)
class WriteResult:
    path: Path
    region_hash: str
    conflict_path: Path | None

## Rendering

In [ ]:
def sanitize_title(title: str, fallback: str) -> str:
    """A filename-safe, Obsidian-link-safe version of the title (rules above)."""
    raise NotImplementedError


def block_id(highlight_id: str) -> str:
    """'h-' + the lowercase ULID (without the caret)."""
    raise NotImplementedError


def render_frontmatter(book: BookMeta) -> str:
    """'---' block: hudhud_book_id, title, then author and language if present
    (each JSON-quoted with ensure_ascii=False), 'tags: [book, hudhud]', '---',
    ending with a newline."""
    raise NotImplementedError


def render_region(book: BookMeta, highlights: list[Highlight], hub_url: str) -> str:
    """START, '## Highlights', a blank line, the highlights sorted by
    (fraction, created_at, id) and separated by blank lines, then END.
    No highlights: the body is '_No highlights yet._'. Per highlight: each text
    line prefixed '> ', a blank line, '^' + block id, a blank line, then
    '*Comment:* ...' (only if the comment is not blank, collapsed to one line)
    and '*Location:* NN% · [Open in reader](<hub_url>/read/<book id>?h=<ULID>)'.
    Escape '<!--' in text and comment. Strip a trailing '/' from hub_url."""
    raise NotImplementedError

## Merging into an existing file

In [ ]:
def extract_region(text: str) -> str | None:
    """The region from START_PREFIX through END (inclusive), after removing a
    leading BOM and normalising CRLF to LF, or None if either marker is missing."""
    raise NotImplementedError


def merge_region(existing: str | None, book: BookMeta, region: str) -> str:
    """New file: frontmatter, blank line, '# <title>', blank, INTRO, blank,
    region, newline. Existing with a region: replace only the region. Existing
    without one: append it after a blank line. Output uses LF line endings."""
    raise NotImplementedError


def region_hash(region: str) -> str:
    """sha256 hex of the region with CRLF normalised and outer whitespace stripped."""
    raise NotImplementedError


def frontmatter_book_id(text: str) -> str | None:
    """The hudhud_book_id value from a leading '---' frontmatter block, if any
    (ignore a leading BOM and CRLF line endings)."""
    raise NotImplementedError

## Writing safely

In [ ]:
def safe_join(root: Path, *parts: str) -> Path:
    """root/parts resolved; ValueError if the result is outside root."""
    raise NotImplementedError


def atomic_write(path: Path, text: str, retries: int = 5) -> None:
    """Write UTF-8 with LF endings to a temp file in path's folder (tempfile.mkstemp),
    fsync, then os.replace. Retry PermissionError up to `retries` times, 0.1 s
    apart. Never leave the temp file behind."""
    raise NotImplementedError


def note_path(vault_root: Path, subfolder: str, book: BookMeta) -> Path:
    """<vault>/<subfolder>/<sanitized title>.md, or '<title> (<id[:8]>).md' if
    the plain name belongs to a different book. Fallback title: id[:12]."""
    raise NotImplementedError


def write_book_note(vault_root: Path, subfolder: str, book: BookMeta,
                    highlights: list[Highlight], hub_url: str,
                    last_hash: str | None, now: str) -> WriteResult:
    """Render and write the note. If the file has a region whose hash differs
    from last_hash (and last_hash is not None), first copy the file to
    '<stem>.conflict-<now without - : and milliseconds>.md'."""
    raise NotImplementedError

## Tests

In [ ]:
BOOK = BookMeta(id="3f2a9c" + "0" * 58, title="Thinking, Fast and Slow",
                author="Daniel Kahneman", language="en")
H42 = Highlight(id="01JB7X3K9Q2M4N5P6R7S8T9V0W",
                text="The confidence people have in their intuitions is not a reliable guide to their validity.",
                comment="Compare with Taleb on narrative fallacy.", fraction=0.42,
                created_at="2026-10-08T10:00:00.000Z")
H10 = Highlight(id="01JB7X0000000000000000000A",
                text="A reliable way to make people believe in falsehoods is frequent repetition.",
                comment="", fraction=0.10, created_at="2026-10-08T11:00:00.000Z")
HUB = "https://hub.example:8765/"
NOW = "2026-10-08T12:34:56.789Z"

GOLDEN = """---
hudhud_book_id: BID
title: "Thinking, Fast and Slow"
author: "Daniel Kahneman"
language: "en"
tags: [book, hudhud]
---

# Thinking, Fast and Slow

Anything written outside the markers is yours. Hudhud never changes it.

<!-- hudhud:start (managed by Hudhud; edit highlights in the app) -->
## Highlights

> A reliable way to make people believe in falsehoods is frequent repetition.

^h-01jb7x0000000000000000000a

*Location:* 10% · [Open in reader](https://hub.example:8765/read/BID?h=01JB7X0000000000000000000A)

> The confidence people have in their intuitions is not a reliable guide to their validity.

^h-01jb7x3k9q2m4n5p6r7s8t9v0w

*Comment:* Compare with Taleb on narrative fallacy.
*Location:* 42% · [Open in reader](https://hub.example:8765/read/BID?h=01JB7X3K9Q2M4N5P6R7S8T9V0W)
<!-- hudhud:end -->
""".replace("BID", BOOK.id)


def write(vault, highlights, last_hash=None, book=BOOK):
    return write_book_note(vault, "Hudhud", book, highlights, HUB, last_hash, NOW)

In [ ]:
%%ipytest -q
import pytest


@pytest.mark.parametrize("title, expected", [
    ("What/If: A?", "What If A"),
    ("CON", "CON_"),
    ("کتاب: نمونه", "کتاب نمونه"),
    ("Note [draft] #1 ^x|y", "Note draft 1 x y"),
    ("Ends with dots...", "Ends with dots"),
    ("...", "FALLBACK"),
    ("x" * 300, "x" * 150),
])
def test_sanitize_title(title, expected):
    assert sanitize_title(title, "FALLBACK") == expected


def test_long_farsi_title_fits_filesystem_limits():
    name = sanitize_title("کتاب " * 100, "FALLBACK")
    assert 0 < len(name.encode("utf-8")) <= 150
    worst = f"{name} (3f2a9c00).conflict-20261008T123456Z.md"
    assert len(worst.encode("utf-8")) <= 255
    assert "\ufffd" not in name


def test_block_id():
    assert block_id("01JB7X3K9Q2M4N5P6R7S8T9V0W") == "h-01jb7x3k9q2m4n5p6r7s8t9v0w"


def test_new_file_matches_golden(tmp_path):
    result = write(tmp_path, [H42, H10])
    assert result.path == tmp_path / "Hudhud" / "Thinking, Fast and Slow.md"
    assert result.path.read_text(encoding="utf-8") == GOLDEN
    assert result.conflict_path is None
    assert result.region_hash == region_hash(extract_region(GOLDEN))


def test_empty_highlights_placeholder(tmp_path):
    text = write(tmp_path, []).path.read_text(encoding="utf-8")
    assert "## Highlights\n\n_No highlights yet._\n" + END in text


def test_multiline_highlight_is_quoted_line_by_line(tmp_path):
    h = Highlight(H10.id, "line one\nline two", "", 0.5, H10.created_at)
    assert "> line one\n> line two\n" in write(tmp_path, [h]).path.read_text(encoding="utf-8")


def test_frontmatter_quotes_yaml_hostile_titles():
    fm = render_frontmatter(BookMeta("x", 'Re: "Kahneman"', None, None))
    assert 'title: "Re: \\"Kahneman\\""' in fm
    assert "author" not in fm

In [ ]:
%%ipytest -q
import pytest


def test_user_text_outside_markers_survives(tmp_path):
    first = write(tmp_path, [H10])
    original = first.path.read_text(encoding="utf-8")
    edited = original.replace(INTRO, INTRO + "\n\nMy own thoughts.") + "\nAfterword.\n"
    first.path.write_text(edited, encoding="utf-8")
    second = write(tmp_path, [H10, H42], last_hash=first.region_hash)
    text = second.path.read_text(encoding="utf-8")
    assert "My own thoughts." in text and text.endswith("Afterword.\n")
    assert "^h-01jb7x3k9q2m4n5p6r7s8t9v0w" in text
    assert second.conflict_path is None


def test_missing_markers_appends_region(tmp_path):
    folder = tmp_path / "Hudhud"
    folder.mkdir()
    (folder / "Thinking, Fast and Slow.md").write_text("# My notes\n\nHand-written.\n",
                                                       encoding="utf-8")
    text = write(tmp_path, [H10]).path.read_text(encoding="utf-8")
    assert text.startswith("# My notes\n\nHand-written.\n\n" + START)
    assert text.endswith(END + "\n")


def test_edited_region_gets_a_conflict_copy(tmp_path):
    first = write(tmp_path, [H10])
    tampered = first.path.read_text(encoding="utf-8").replace("frequent repetition", "MY EDIT")
    first.path.write_text(tampered, encoding="utf-8")
    second = write(tmp_path, [H10], last_hash=first.region_hash)
    assert second.conflict_path == first.path.with_name(
        "Thinking, Fast and Slow.conflict-20261008T123456Z.md")
    assert second.conflict_path.read_text(encoding="utf-8") == tampered
    assert "frequent repetition" in second.path.read_text(encoding="utf-8")


def test_crlf_only_change_is_not_a_conflict(tmp_path):
    first = write(tmp_path, [H10])
    first.path.write_bytes(first.path.read_bytes().replace(b"\n", b"\r\n"))
    assert write(tmp_path, [H10], last_hash=first.region_hash).conflict_path is None


def test_no_known_hash_means_no_conflict(tmp_path):
    first = write(tmp_path, [H10])
    first.path.write_text(first.path.read_text("utf-8").replace("frequent", "X"), "utf-8")
    assert write(tmp_path, [H10], last_hash=None).conflict_path is None


def test_marker_text_in_highlight_is_escaped(tmp_path):
    evil = Highlight(H10.id, "before <!-- hudhud:end --> after", "see <!-- hudhud:start", 0.5,
                     H10.created_at)
    text = write(tmp_path, [evil]).path.read_text(encoding="utf-8")
    assert text.count(END) == 1 and text.count(START_PREFIX) == 1
    assert "&lt;!-- hudhud:end --> after" in extract_region(text)


def test_same_title_different_books_do_not_collide(tmp_path):
    other = BookMeta("9" * 64, BOOK.title, "Someone Else", "en")
    a = write(tmp_path, [H10])
    b = write(tmp_path, [H42], book=other)
    assert a.path != b.path
    assert b.path.name == "Thinking, Fast and Slow (99999999).md"
    assert frontmatter_book_id(a.path.read_text("utf-8")) == BOOK.id
    assert write(tmp_path, [H10]).path == a.path


def test_bom_does_not_hide_ownership(tmp_path):
    a = write(tmp_path, [H10])
    a.path.write_bytes(b"\xef\xbb\xbf" + a.path.read_bytes())
    assert frontmatter_book_id(a.path.read_text("utf-8")) == BOOK.id
    other = BookMeta("9" * 64, BOOK.title, None, None)
    assert write(tmp_path, [H42], book=other).path != a.path


def test_safe_join_blocks_escape(tmp_path):
    with pytest.raises(ValueError):
        safe_join(tmp_path, "..", "outside.md")
    with pytest.raises(ValueError):
        write_book_note(tmp_path, "../outside", BOOK, [H10], HUB, None, NOW)


def test_atomic_write_leaves_no_temp_files(tmp_path):
    target = tmp_path / "note.md"
    atomic_write(target, "a\nb\n")
    atomic_write(target, "c\n")
    assert [p.name for p in tmp_path.iterdir()] == ["note.md"]
    assert target.read_bytes() == b"c\n"

## Graduate

1. `hub/src/hudhud_hub/vault.py`: imports (minus `ipytest`), constants, the
   three dataclasses and your functions.
2. `hub/tests/test_vault.py`: the fixture and both test cells, with
   `from hudhud_hub.vault import *`. The golden string doubles as the
   golden-file test from spec section 13.
3. `uv run --project hub pytest hub/tests/test_vault.py -q`

Try it on your real vault: call `write_book_note` with your vault path and a
couple of made-up highlights, open Obsidian, and click a block link.